In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.optimize import curve_fit

# -------------------------------
# Model: Gaussian + linear background
# -------------------------------
def gauss_lin(x, A, mu, sigma, m, b):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + m * x + b

# -------------------------------
# Load your data
# Replace this with your own file
# -------------------------------
# Example: two-column file (channel, counts)
data = np.loadtxt("spectrum.txt")
x = data[:, 0]
y = data[:, 1]

# -------------------------------
# Peak detection
# -------------------------------
peaks, properties = find_peaks(
    y,
    height=max(y)*0.05,     # threshold (5% of max)
    distance=10,            # minimum separation
    prominence=10           # helps avoid noise
)

print(f"Found {len(peaks)} peaks")

# -------------------------------
# Fit each peak
# -------------------------------
fit_results = []

for peak in peaks:
    # Define a local window around the peak
    window = 20  # adjust depending on resolution
    left = max(0, peak - window)
    right = min(len(x), peak + window)

    x_fit = x[left:right]
    y_fit = y[left:right]

    # Initial guesses
    A0 = y[peak] - np.min(y_fit)
    mu0 = x[peak]
    sigma0 = 2.0
    m0 = 0
    b0 = np.min(y_fit)

    p0 = [A0, mu0, sigma0, m0, b0]

    try:
        popt, pcov = curve_fit(gauss_lin, x_fit, y_fit, p0=p0)

        fit_results.append((popt, pcov))

        # Plot fit
        x_dense = np.linspace(x_fit[0], x_fit[-1], 200)
        y_dense = gauss_lin(x_dense, *popt)

        plt.plot(x_dense, y_dense, 'r--')

        print(f"Peak at {mu0:.2f} fitted:")
        print(f"  A = {popt[0]:.2f}")
        print(f"  mu = {popt[1]:.2f}")
        print(f"  sigma = {popt[2]:.2f}")
        print(f"  m = {popt[3]:.4f}")
        print(f"  b = {popt[4]:.2f}")
        print()

    except RuntimeError:
        print(f"Fit failed for peak at {mu0:.2f}")

# -------------------------------
# Plot everything
# -------------------------------
plt.figure(figsize=(10, 6))
plt.plot(x, y, label="Spectrum")
plt.plot(x[peaks], y[peaks], "x", label="Detected Peaks")

plt.xlabel("Channel / Energy")
plt.ylabel("Counts")
plt.legend()
plt.title("Gamma Spectrum Peak Fitting")

plt.show()